# Final PCA: covariates for phenotype residualization

Round 1/2/2b's PCAs all exist to *classify ancestry* -- fit on 1000G-comparable variants (`agreeing_snps.ids`, round 1's ID+REF+ALT-verified, HM3-restricted set) so AoU samples can be projected into a space where 1000G population labels are meaningful, and (round 2/2b) refit tighter for a purity threshold. That HM3 restriction is necessary for classification, but it's a real cost once the ancestry decision is already made: HM3 is ascertainment-biased toward common EUR variants (chosen for the original HapMap project), which distorts PC resolution for non-EUR cohorts -- most concretely the `afr` sample set -- and its ~1.3M-variant density is much lower than what's actually available.

This notebook is a *separate, later* step: once a sample set's ancestry-filtered cohort is fixed (round 2b's keep-list), it fits a fresh PCA on `genome_wide_qc_thinning_merge.ipynb`'s already-built panel (also in `01_ancestry_filtering`) -- QC'd and LD-pruned from the **full ACAF catalog**, not HM3-restricted. That panel is general-purpose, not built for this notebook specifically -- `03_grm_shards`'s GRM construction reads its bed export, this notebook reads its pgen export -- so no new QC/pruning work is needed here. It IS further thinned below, though: that panel's ~1M-variant density is what GRM relatedness estimation wants, but PCA for population-structure covariates converges with far fewer, well-spaced markers -- ~100K is standard practice -- so a second `--thin` pass (same calibrate-then-apply pattern `king_po_exclusion.ipynb` already uses for its own kinship-specific thinning) cuts the PCA fit down to a lighter, still-representative SNP set before `--pca approx` runs.

No 1000G projection here -- that's the point: this PCA's only job is producing good covariates for `residualize_phenotypes.ipynb`, not classifying anyone, so there's no need to keep it on a reference-comparable variant set. `--pca approx` (Galinsky et al. 2016, "fastPCA"), same as `reverse_pca_aou.ipynb`, since this runs on the full round-2b-passing cohort, not a subsample.

Depends on `genome_wide_qc_thinning_merge.ipynb` having already been run for this `SAMPLE_SET` -- this notebook only reads that panel, it doesn't build it. Both notebooks live in `01_ancestry_filtering` (not `03_grm_shards`) since the panel itself is ancestry-filtering-adjacent general-purpose infrastructure, not GRM-specific -- `03_grm_shards`'s notebooks are a second, independent consumer of the same build.

## Compute resource

Lighter than `03_grm_shards/grm_shard_timing.ipynb`/`grm_shard_run.ipynb` size for -- this reads the same panel initially, but thins it down to `PCA_N_SNPS_TARGET` (~100K) before the actual PCA fit, so the CPU/I/O-heavy part is much smaller than a full ~1M-variant pass. Still worth sizing up from Workbench 2.0's small default (8-16 vCPU is plenty) rather than assuming the default 2 CPU / 13 GB handles even the initial copy/thin comfortably.

## Setup

plink2: manual install, same pattern as everywhere else in this repo.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version
nproc
free -h

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

N_THREADS = os.cpu_count()

## Inputs

`MERGED_PREFIX` points at `genome_wide_qc_thinning_merge.ipynb`'s pgen output for this `CDR_VERSION`/`SAMPLE_SET` (`genome_wide_round2b_thinned_{CDR_VERSION}_{SAMPLE_SET}` -- the pgen form, not the `_bed` PLINK1 export that notebook also writes for `03_grm_shards`'s GRM step; plink2 reads pgen natively via `--pfile`, no need for the bed conversion here). Copied to local scratch first, same convention as everywhere else in this pipeline -- plink2 reading a multi-GB panel repeatedly over the gcsfuse-mounted bucket is much slower than local disk.

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)

# Must match genome_wide_qc_thinning_merge.ipynb's CDR_VERSION/SAMPLE_SET -- this
# notebook only reads that notebook's merged panel, it doesn't build one.
CDR_VERSION = "v9"
SAMPLE_SET = "eur"   # <-- change this and rerun for each of the 5 sample sets

# same directory genome_wide_qc_thinning_merge.ipynb writes to -- both notebooks
# live in 01_ancestry_filtering, so this is a same-directory read, not a
# cross-stage one (unlike 03_grm_shards' GRM notebooks, which read this same
# panel's bed export from here too)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/{CDR_VERSION}/01_ancestry_filtering/genome_wide_panel_{SAMPLE_SET}"
FINAL_PCA_BUCKET_DIR = f"{BUCKET_DIR}/final_pca"
os.makedirs(FINAL_PCA_BUCKET_DIR, exist_ok=True)

MERGED_NAME = f"genome_wide_round2b_thinned_{CDR_VERSION}_{SAMPLE_SET}"   # pgen form, from genome_wide_qc_thinning_merge.ipynb

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")   # same scratch dir genome_wide_qc_thinning_merge.ipynb/grm_shard_timing.ipynb use -- panel may already be there
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, MERGED_NAME)

for ext in ("pgen", "pvar", "psam"):
    bucket_path = f"{BUCKET_DIR}/{MERGED_NAME}.{ext}"
    local_path = f"{MERGED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), (
        f"missing merged panel: {bucket_path!r} -- run genome_wide_qc_thinning_merge.ipynb's "
        f"merge section for SAMPLE_SET={SAMPLE_SET!r} first"
    )
    if not os.path.isfile(local_path):
        import shutil
        shutil.copy(bucket_path, local_path)

# GRM relatedness estimation wants the dense ~1M-variant panel MERGED_PREFIX points
# at; PCA for population-structure covariates converges with far fewer, well-spaced
# markers -- ~100K is standard practice -- so PCA_PREFIX below fits on a further-
# thinned subset of it instead of the full panel (see "Further thin for PCA" below).
PCA_N_SNPS_TARGET = 100_000
PCA_BED_PREFIX = os.path.join(LOCAL_WORK_DIR, f"{MERGED_NAME}_pca_thinned")

N_PCS = 20   # matches every other PC output in this pipeline (round 2b's PC_COVARIATE_PATH, etc.)
FINAL_PCA_PREFIX = os.path.join(LOCAL_WORK_DIR, f"final_pca_{CDR_VERSION}_{SAMPLE_SET}")

print(MERGED_PREFIX)
print(FINAL_PCA_BUCKET_DIR)

## Further thin for PCA

`--thin <p>` (plink2: keep each variant independently with probability `p`), solved for the `p` that hits `PCA_N_SNPS_TARGET` from the panel's current count -- same calibrate-then-apply pattern `chr22_qc_thinning_timing.ipynb` uses for its own genome-wide target and `king_po_exclusion.ipynb` uses to thin this same panel down further for KING specifically (50K there, for kinship/IBS0 estimation -- a different downstream need with its own target). Random thinning on top of an already-random `--thin` doesn't introduce correlation, only removes density further, so this stays a reasonable approximately-independent SNP set for PCA purposes.

In [ ]:
%%bash -s "$MERGED_PREFIX" "$PCA_BED_PREFIX" "$PCA_N_SNPS_TARGET" "$N_THREADS"
set -e
MERGED_PREFIX=$1
PCA_BED_PREFIX=$2
N_TARGET=$3
THREADS=$4

if [ -s "${PCA_BED_PREFIX}.pgen" ]; then
  echo "already thinned, skipping"
else
  N_CURRENT=$(($(wc -l < "${MERGED_PREFIX}.pvar") - 1))
  THIN_P=$(python3 -c "print(min(1.0, ${N_TARGET} / ${N_CURRENT}))")
  echo "current SNPs: $N_CURRENT, target: $N_TARGET, thin_p: $THIN_P"

  plink2 \
    --pfile "$MERGED_PREFIX" \
    --thin "$THIN_P" \
    --threads "$THREADS" \
    --make-pgen \
    --out "$PCA_BED_PREFIX"
fi

echo "Thinned SNP count:"
awk 'END{print NR-1}' "${PCA_BED_PREFIX}.pvar"

## Fit the PCA

`--pca approx`: fits on the full round-2b-passing cohort directly (no subsampling), same reasoning as `reverse_pca_aou.ipynb` -- plink2's exact PCA algorithm doesn't scale to a cohort this size, so this uses the randomized/Blanczos algorithm (Galinsky et al. 2016, "fastPCA") instead, now on the ~100K-variant `PCA_BED_PREFIX` from above rather than the full ~1M-variant GRM panel. No `--keep` needed -- the panel is already restricted to round 2b's keep-list (`ROUND2B_KEEP_PATH` in `genome_wide_qc_thinning_merge.ipynb`), and no HWE/LD-pruning here either -- both already applied when that panel was built, so this cell is just the PCA fit itself, not a QC pass.

In [ ]:
%%bash -s "$PCA_BED_PREFIX" "$FINAL_PCA_PREFIX" "$N_THREADS" "$N_PCS"
set -e
PCA_BED_PREFIX=$1
FINAL_PCA_PREFIX=$2
THREADS=$3
NPCS=$4

plink2 \
  --pfile "$PCA_BED_PREFIX" \
  --nonfounders \
  --freq counts \
  --pca approx "$NPCS" \
  --threads "$THREADS" \
  --out "$FINAL_PCA_PREFIX"

ls -lh "${FINAL_PCA_PREFIX}".*

## Scree plot

Quick sanity check -- % variance explained per PC. A PCA fit on a much denser, unbiased-by-HM3-ascertainment panel should still show the expected steep drop-off after the first few structure-carrying PCs; a flat/noisy scree here would be a sign something's off in the input panel or the thinning target, not necessarily in this fit.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

eigenval = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenval", header=None, names=["eigenvalue"])
eigenval["pc"] = range(1, len(eigenval) + 1)
eigenval["pct_variance"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(eigenval["pc"], eigenval["pct_variance"], color="royalblue")
ax.set_xlabel("PC")
ax.set_ylabel("% variance explained")
ax.set_title(f"Final PCA [{SAMPLE_SET}] scree plot")
ax.set_xticks(eigenval["pc"])
plt.tight_layout()
plot_path = os.path.join(FINAL_PCA_BUCKET_DIR, f"final_pca_scree_{SAMPLE_SET}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")
print(eigenval[["pc", "eigenvalue", "pct_variance"]].to_string(index=False))

## Write PC covariates for residualize_phenotypes.ipynb

Same `IID PC1 ... PC20` format `residualize_phenotypes.ipynb`'s `PC_PATH` / `pull_covariates()` expects (and the same format `residualize_phenotypes_round2.ipynb`'s "Build round 2 PC covariates" cell produces) -- plink2's direct `.eigenvec` output already has one row per sample, just needs the `#FID`/`FID` column dropped and the ID column normalized to `IID`.

In [ ]:
direct = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenvec", sep=r"\s+")

id_col = "#IID" if "#IID" in direct.columns else "IID"
pc_cols = [c for c in direct.columns if c.startswith("PC")]
assert len(pc_cols) == N_PCS, f"expected {N_PCS} PC columns, found {len(pc_cols)}: {pc_cols}"

covariate_table = direct[[id_col] + pc_cols].rename(columns={id_col: "IID"})
covariate_table["IID"] = covariate_table["IID"].astype(str)

PC_COVARIATE_PATH = os.path.join(FINAL_PCA_BUCKET_DIR, f"final_pca_pc_covariates_{SAMPLE_SET}.txt")
covariate_table.to_csv(PC_COVARIATE_PATH, sep="\t", index=False)
print(f"Wrote {len(covariate_table)} samples' PC1-PC{N_PCS} -> {PC_COVARIATE_PATH}")

## Next steps

Not wired into a residualization notebook yet -- `residualize_phenotypes.ipynb`/`residualize_phenotypes_round2.ipynb` still point `PC_PATH` at round 2b's/round 2's own covariate output. To use this notebook's PCs instead, either point one of those notebooks' `PC_PATH` at `PC_COVARIATE_PATH` above (keeping `KEEP_LIST_PATH` as-is, since the sample set itself is unchanged -- only the PC covariates differ), or add a third `residualize_phenotypes_final_pca.ipynb` twin, same pattern as the round-2 one, once this fit has been checked out for real.